# 02 · Perfilado del crudo Cinemometros (SUTRAN)

Segunda fuente, deliberadamente distinta: 160.018 filas en CSV frente a un XLSX
de 9.106 filas. Aqui los problemas son de **normalizacion**, no de formato:

- `REGION` con `LIMA` y `Lima` a la vez (doble caja)
- `CARRETERA` con `Panamericana Sur` frente a `PANAMERICANA SUR`
- `lat`/`lon` que duplican exactamente `LATITUD`/`LONGITUD`
- Fechas en formato numerico compactado (`20191031`)

Al ser un dataset mucho mayor, se mide tambien el coste de la normalizacion.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

from src.utils.configloader import expand_path, load_catalogs, load_settings, load_sources

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

SETTINGS = load_settings()
spec = load_sources()["cinemometros"].copy()
spec["path"] = expand_path(spec["path"])
print("Fuente:", spec["path"])
print("Existe:", Path(spec["path"]).exists())

In [ ]:
import csv

with open(spec['path'], encoding='utf-8-sig') as fh:
    reader = csv.reader(fh)
    header = next(reader)
    first = next(reader)
print("Columnas:", len(header))
for h in header:
    print('   ', repr(h))
print('\nPrimera fila:'); print(first)

## 2.1 · La doble caja en REGION

El problema de normalizacion clasico: la misma entidad escrita de dos formas.

In [ ]:
cine = pd.read_csv(spec["path"], encoding="utf-8-sig", low_memory=False)
print(f"Crudo: {cine.shape[0]:,} filas x {cine.shape[1]} columnas\n")
print("Valores de REGION tal cual llegan:")
print(cine['REGION'].value_counts(dropna=False).to_string())

### Cuantas filas se ven afectadas por la doble caja

In [ ]:
es_minuscula = cine["REGION"].astype(str) != cine["REGION"].astype(str).str.upper()
print("Filas con REGION no normalizada:", f"{int(es_minuscula.sum()):,}",
      f"({es_minuscula.mean()*100:.2f}%)")
print('Valores afectados:', sorted(cine.loc[es_minuscula, 'REGION'].unique()))
print("\n=> Sin case_normalize, el mismo departamento seria dos categorias distintas.")

## 2.2 · CARRETERA tambien tiene doble caja

In [ ]:
print("Valores de CARRETERA:")
print(cine['CARRETERA'].value_counts(dropna=False).head(20).to_string())

## 2.3 · Fechas en formato numerico compactado

`20191031` es un entero, no una fecha. Sin `date_columns` con `%Y%m%d` el pipeline
no puede parsearlas y la regla critica `fecha_parsable` falla.

In [ ]:
print("Dtype crudo de FECHA_PAPELETA:", cine["FECHA_PAPELETA"].dtype)
print('Muestra:', cine['FECHA_PAPELETA'].head(5).tolist(), '\n')
parseada = pd.to_datetime(cine["FECHA_PAPELETA"], format="%Y%m%d", errors="coerce")
print("NaT con format=%Y%m%d:", int(parseada.isna().sum()))
print("Rango:", parseada.min(), "a", parseada.max())
print("\n=> Configurar el formato compacto es obligatorio, no opcional.")

## 2.4 · Columnas redundantes: lat/lon vs LATITUD/LONGITUD

La configuracion descarta `lat`/`lon`. Este notebook verifica que la duplicacion es
exacta antes de tomar esa decision.

In [ ]:
if "lat" in cine.columns and "lon" in cine.columns:
    mismo_lat = (cine['lat'].astype(float) == cine['LATITUD'].astype(float)).mean()
    mismo_lon = (cine['lon'].astype(float) == cine['LONGITUD'].astype(float)).mean()
    print(f"Pct de filas donde lat == LATITUD: {mismo_lat*100:.2f}%")
    print(f"Pct de filas donde lon == LONGITUD: {mismo_lon*100:.2f}%")
    print("\n=> Duplicacion exacta: se descartan en clean (drop_columns).")
else:
    print("El fichero ya no trae lat/lon.")

## 2.5 · Nulos y completitud

In [ ]:
nulos = pd.DataFrame({
    'nulos': cine.isna().sum(),
    'pct_nulos': (cine.isna().mean() * 100).round(2),
    'n_unicos': cine.nunique(dropna=True),
}).sort_values('pct_nulos', ascending=False)
nulos[nulos['pct_nulos'] > 0]

## 2.6 · Velocidades: el dominio de negocio

El exceso de velocidad es la magnitud accionable. Se derivan `exceso_kmh` y
`excede_limite` en `src/transform/features.py`.

In [ ]:
v = pd.to_numeric(cine["VELOCIDAD_DETECTADA"], errors="coerce")
lim = pd.to_numeric(cine["LIMITE_VELOCIDAD"], errors="coerce")
v, lim = v.dropna(), lim.loc[v.dropna().index]
print("Velocidad detectada:")
print(v.describe().round(2).to_string(), '\n')
print("Limite de velocidad:")
print(lim.value_counts(dropna=False).head(10).to_string(), '\n')
exceso = (v - lim).clip(lower=0)
print("Exceso sobre el limite (km/h):")
print(exceso.describe().round(2).to_string())
print(f"\nPct que excede el limite: {(exceso > 0).mean()*100:.2f}%")
print("Pct que excede mas de 20 km/h: %.2f%%" % ((exceso > 20).mean()*100))

### Outliers: se reportan, no se eliminan

`outlier_policy: report`. Un exceso alto es informacion real, no basura.

In [ ]:
q1, q3 = exceso.quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
print(f"IQR: Q1={q1:.1f}, Q3={q3:.1f}, umbral superior={lim_sup:.1f} km/h")
print(f"Outliers por IQR: {int((exceso > lim_sup).sum()):,} ({(exceso > lim_sup).mean()*100:.2f}%)")
print("\n=> Se reportan en el perfil. Borrarlos seria una decision analitica, no de limpieza.")

## 2.7 · Duplicados en la clave primaria

In [ ]:
clave = "NRO_DETECCION"
print("Filas         :", f"{len(cine):,}")
print("Detecciones   :", f"{cine[clave].nunique():,}")
print("Duplicados    :", int(cine[clave].duplicated().sum()))
print("Nulos en la PK:", int(cine[clave].isna().sum()))
if cine[clave].duplicated().any():
    d = cine[cine[clave].duplicated(keep=False)]
    print("\nMuestra de duplicados:")
    print(d.head(10).to_string())
else:
    print("\nPK unica: no se requiere dedupe en este dataset.")

## 2.8 · Distribucion geografica

In [ ]:
print('Top 10 regiones por numero de detecciones:')
print(cine['REGION'].value_counts().head(10).to_string())

## 2.9 · Conclusion: contraste con ONSV

| Aspecto | ONSV (9.106 filas) | Cinemometros (160.018 filas) |
|---|---|---|
| Formato | XLSX con preambulo | CSV plano |
| Problema dominante | estructura (nombres, tipos) | normalizacion (caja, acentos) |
| Fechas | `DD/MM/YYYY` ambiguo | `%Y%m%d` numerico |
| Columnas redundantes | ninguna | `lat`/`lon` duplican |
| PK | `codigo`, unica | `nro_deteccion`, unica |
| Nulos relevantes | `vehiculos_danados` 30.9% | ninguno critico |

Un pipeline que solo funciona con uno de los dos formatos no es un pipeline.

## 2.10 · Visualización: exceso de velocidad

La magnitud accionable del dataset. El histograma muestra una masa junto al
límite y una cola larga: la cola es donde está el riesgo vial.
De nuevo: los outliers **no se borran**; se reportan y se estudian como señal.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'savefig.bbox': 'tight'})
PALETTE = ['#4f46e5', '#059669', '#d97706', '#dc2626', '#0891b2']

# Mostrar en línea y capturar cualquier figura como imagen embebida.
from IPython.display import Image as _IPImage, display
import base64 as _b64
from pathlib import Path as _Path

def _show(png_path):
    """Muestra un PNG en línea dentro del notebook."""
    p = _Path(png_path)
    if not p.exists():
        print('Falta', p, '- ejecuta python scripts/graficos_etl.py')
        return
    display(_IPImage(data=_b64.b64decode(_b64.b64encode(p.read_bytes()))))
exceso = (v - lim).clip(lower=0)
print('Exceso sobre el límite (km/h):')
print(exceso.describe().round(2).to_string())
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].hist(exceso, bins=40, color=PALETTE[0], edgecolor='white')
ax[0].axvline(lim.median(), color='#dc2626', ls='--', label='límite mediano')
ax[0].set_title('Distribución del exceso de velocidad')
ax[0].set_xlabel('km/h por encima del límite')
ax[0].legend()
sns.boxplot(x=exceso, ax=ax[1], color=PALETTE[0])
ax[1].set_title('Boxplot: cola derecha = exceso grave')
plt.tight_layout()
plt.show()

In [ ]:
q1, q3 = exceso.quantile([0.25, 0.75])
iqr = q3 - q1
umbral = q3 + 1.5 * iqr
print(f'IQR = {iqr:.1f} | umbral superior = {umbral:.1f} km/h')
print(f'Outliers: {int((exceso > umbral).sum()):,} ({(exceso > umbral).mean()*100:.2f}%)')
plt.figure(figsize=(9, 4.5))
sns.histplot(exceso, bins=40, kde=True, color=PALETTE[0], stat='density')
plt.axvline(umbral, color='#dc2626', ls='--', label=f'umbral IQR = {umbral:.1f}')
plt.title('Exceso de velocidad con umbral IQR')
plt.legend()
plt.tight_layout()
plt.show()

## 2.11 · Velocidad por carretera y por región

No todos los puntos de control son iguales: si una carretera concentra exceso,
ahí hay más riesgo de accidente. Esta es la lectura que justifica el módulo de
"ruta segura" del SIPAT.

In [ ]:
por_carretera = (cine.assign(_e=exceso.reset_index(drop=True))
                   .groupby('CARRETERA', dropna=False)['_e']
                   .agg(['count', 'mean']).sort_values('mean', ascending=False))
por_carretera['exceso_medio'] = por_carretera['mean'].round(2)
top = por_carretera.head(10)
print('Exceso medio por carretera (top 10):')
print(top[['count', 'exceso_medio']].to_string())
plt.figure(figsize=(9, 4.5))
sns.barplot(data=top.reset_index(), x='CARRETERA', y='exceso_medio', color=PALETTE[2])
plt.xticks(rotation=35, ha='right')
plt.axhline(exceso.mean(), color='#dc2626', ls='--', label=f'media global = {exceso.mean():.2f}')
plt.ylabel('exceso medio (km/h)')
plt.title('Exceso medio por carretera')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
por_region = (cine.assign(_e=exceso.reset_index(drop=True))
                  .groupby('REGION', dropna=False)['_e']
                  .agg(['count', 'mean']).sort_values('mean', ascending=False))
por_region['exceso_medio'] = por_region['mean'].round(2)
top = por_region.head(12)
plt.figure(figsize=(9, 4.5))
sns.barplot(data=top.reset_index(), x='REGION', y='exceso_medio', color=PALETTE[0])
plt.xticks(rotation=35, ha='right')
plt.axhline(exceso.mean(), color='#dc2626', ls='--', label='media global')
plt.ylabel('exceso medio (km/h)')
plt.title('Exceso medio por región (los nombres ya vienen normalizados por el ETL)')
plt.legend()
plt.tight_layout()
plt.show()